# Pipeline Diagnostic & Debugger
Use this to verify the `compute_altcoin_features` signature and data integrity.

In [1]:
import os, sys, pandas as pd
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)

import config
from utils.data_utils import update_all_tickers
from utils.features import compute_anchor_features, compute_macro_risk_state, compute_altcoin_features

print("--- Step 1: Fetching Data ---")
dfs = update_all_tickers()
c_dfs = {k: v for k, v in dfs.items() if '/' in k or ':' in k}
m_dfs = {k: v for k, v in dfs.items() if not ('/' in k or ':' in k)}

print("--- Step 2: Computing Anchors ---")
ms = compute_macro_risk_state(m_dfs)
btc_a = compute_anchor_features(c_dfs[config.ANCHOR_TICKERS[0]], 'BTC')
eth_a = compute_anchor_features(c_dfs[config.ANCHOR_TICKERS[1]], 'ETH')

anchors_dict = {'BTC': btc_a, 'ETH': eth_a}

print(f"Macro State Type: {type(ms)}")
print(f"Anchors Dict Keys: {list(anchors_dict.keys())}")

print("\n--- Step 3: Testing compute_altcoin_features Signature ---")
test_ticker = config.ALTCOIN_TICKERS[0]
try:
    print(f"Attempting 3-argument call for {test_ticker}...")
    feat_all = compute_altcoin_features(c_dfs[test_ticker], anchors_dict, ms)
    print(f"✅ SUCCESS! Features shape: {feat_all.shape}")
except TypeError as e:
    print(f"❌ 3-argument call FAILED: {e}")
    
try:
    print(f"\nChecking for old 4-argument call signature...")
    feat_all_old = compute_altcoin_features(c_dfs[test_ticker], btc_a, eth_a, ms)
    print("✅ SUCCESS! (Wait, this means your features.py is still the old version)")
except TypeError as e:
    print(f"❌ 4-argument call FAILED (Expected): {e}")

--- Step 1: Fetching Data ---
--- Step 2: Computing Anchors ---
Macro State Type: <class 'pandas.Series'>
Anchors Dict Keys: ['BTC', 'ETH']

--- Step 3: Testing compute_altcoin_features Signature ---
Attempting 3-argument call for BNB/USDT:USDT...
✅ SUCCESS! Features shape: (17534, 66)

Checking for old 4-argument call signature...
❌ 4-argument call FAILED (Expected): compute_altcoin_features() takes 3 positional arguments but 4 were given
